In [ ]:
# Cell 1 — Setup
import os, sys, copy
from pathlib import Path
os.chdir(Path(r"C:\Users\amand\OneDrive - Fisk University\Documents\ResumeBiasResearch"))
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.models.train import get_models, evaluate_classifier
from src.fairness.metrics import compute_all
from src.fairness.intervention import compute_reweighting_weights, infer_sensitive_from_names

sns.set_style("whitegrid")
data = np.load('data/processed/prepared_data.npz')
X_train, X_test = data['X_train'], data['X_test']

raw = np.load('data/raw/FairCVdb.npy', allow_pickle=True).item()
names_train = raw['Names Train']
names_test = raw['Names Test']

sensitive_train = infer_sensitive_from_names(names_train)
sensitive_test = infer_sensitive_from_names(names_test)
print(pd.Series(sensitive_train).value_counts())

In [ ]:
# Cell 2 — Baseline (no intervention) on gender-biased labels
y_train = data['y_train_gender']
y_test = data['y_test_gender']

baseline_results = []
for name, model_proto in get_models().items():
    m = copy.deepcopy(model_proto)
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    perf = evaluate_classifier(m, X_test, y_test)
    fair = compute_all(y_test, y_pred, sensitive_test)
    baseline_results.append({
        'model': name,
        'stage': 'baseline',
        'accuracy': perf['accuracy'], 'f1': perf['f1'], 'auc': perf['auc'],
        'dpd': fair['demographic_parity_difference'],
        'eod': fair['equal_opportunity_difference'],
        'dir_min': min(fair['disparate_impact_ratios'].values()),
    })

print(pd.DataFrame(baseline_results).to_string(index=False))

In [ ]:
# Cell 3 — After reweighting
weights = compute_reweighting_weights(y_train, sensitive_train)

intervened_results = []
for name, model_proto in get_models().items():
    m = copy.deepcopy(model_proto)
    m.fit(X_train, y_train, sample_weight=weights)
    y_pred = m.predict(X_test)
    perf = evaluate_classifier(m, X_test, y_test)
    fair = compute_all(y_test, y_pred, sensitive_test)
    intervened_results.append({
        'model': name,
        'stage': 'reweighted',
        'accuracy': perf['accuracy'], 'f1': perf['f1'], 'auc': perf['auc'],
        'dpd': fair['demographic_parity_difference'],
        'eod': fair['equal_opportunity_difference'],
        'dir_min': min(fair['disparate_impact_ratios'].values()),
    })

print(pd.DataFrame(intervened_results).to_string(index=False))

In [ ]:
# Cell 4 — Before/after comparison table
df = pd.DataFrame(baseline_results + intervened_results)
df.to_csv('results/tables/fairness_intervention_comparison.csv', index=False)

# Visualize before/after
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(
    axes,
    ['accuracy', 'dpd', 'eod'],
    ['Accuracy', 'Demographic Parity Difference\n(lower is better)',
     'Equal Opportunity Difference\n(lower is better)']
):
    df.pivot(index='model', columns='stage', values=metric).plot(kind='bar', ax=ax)
    ax.set_title(title)
    ax.set_ylabel(metric)

plt.tight_layout()
plt.savefig('results/figures/05_intervention_before_after.png', dpi=100, bbox_inches='tight')
plt.show()